# 01 — Ingest

**Project: billboard-hot100** — how have music **genres** risen and fallen
at the top of the Billboard Hot 100 over the decades?

Two sources, both keyless:
1. **Billboard Hot 100 full chart history** — every weekly chart 1958→present,
   flattened to one row per (date, song, artist) appearance.
2. **MusicBrainz artist tags** — Billboard carries NO genre, so genre is derived
   at the **artist** level from MusicBrainz crowd-sourced tags.

This stage lands **RAW data only** — download + flatten + an artist→tags lookup.
No typing, dedupe, artist normalization, or genre bucketing happens here; all of
that is `02-clean`'s job. The messy raw tags are stored verbatim.

In [ ]:
import sys, os
from pathlib import Path

# Navigate to project root regardless of how the kernel was launched
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import (
    load_config, ingest_hot100, artist_appearance_counts,
    ingest_musicbrainz_genres,
)
from src.clean_quality import get_connection, load_to_duckdb, register_source

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Source 1 — Billboard Hot 100 full chart history

Source: **[mhollingshead/billboard-hot-100](https://github.com/mhollingshead/billboard-hot-100)**
— a daily-updated crawl of Billboard.com's Hot 100, published as a single
~44 MB JSON array on raw-GitHub (keyless, no auth). The underlying chart is
© Billboard; used here for non-commercial research.

Each element is one weekly chart:
```json
{"date": "1958-08-04",
 "data": [{"song", "artist", "this_week", "last_week",
           "peak_position", "weeks_on_chart"}, ... up to 100 rows ...]}
```

`ingest_hot100()` downloads the file **once** (cached to
`data/raw/hot100_all.json` — a 44 MB file isn't re-pulled every run) and
**flattens** it to one row per (date, song, artist) chart appearance. This is
a flatten, NOT a transform: `this_week` (the rank that week), `last_week`,
`peak_position`, and `weeks_on_chart` are kept exactly as the source stores
them. Expect ~355k appearances.

In [ ]:
hot100 = ingest_hot100(cfg)
print(f'\nrows: {len(hot100):,}')
print(f'date range: {hot100["date"].min()} -> {hot100["date"].max()}')
hot100.head(10)

### Load the flattened long table into DuckDB as `hot100_raw`
Lands untouched (one row per chart appearance) and registers provenance.
**Series-break note:** Billboard's Hot 100 methodology changed over the decades
(sales + radio airplay historically; digital downloads folded in 2005, then
streaming from 2007 and especially the 2010s on-demand/YouTube-era revisions) —
a real consideration for any over-time comparison, carried in `_sources`.

In [ ]:
load_to_duckdb(hot100, 'hot100_raw', con)
n_hot = con.execute('SELECT COUNT(*) FROM hot100_raw').fetchone()[0]
print(f'hot100_raw: {n_hot:,} rows')

register_source(
    con, 'hot100_raw',
    'Billboard Hot 100 full chart history (mhollingshead/billboard-hot-100, GitHub)',
    url=cfg['sources']['hot100_all']['url'],
    license=cfg['sources']['hot100_all']['license'],
    notes=(
        'Every weekly Hot 100 chart 1958-08-04 -> present, flattened to one row '
        'per (date, song, artist) appearance. Fields verbatim from source: '
        'this_week (rank that week), last_week, peak_position, weeks_on_chart. '
        'Underlying chart (c) Billboard; GitHub crawl used for non-commercial research.'
    ),
    methodology=(
        'A crawl of Billboard.com Hot 100 published as one JSON array; no '
        'transformation applied here beyond exploding the nested weekly data lists.'
    ),
    series_breaks=(
        'Billboard Hot 100 ranking methodology shifted over the decades: sales + '
        'radio airplay historically, digital downloads incorporated from 2005, '
        'online streaming from 2007 and progressively weighted through the 2010s '
        '(YouTube/on-demand). Chart presence is therefore NOT measured the same way '
        'across eras -- any genre-over-time comparison must note this.'
    ),
)
print('registered hot100_raw in _sources')

## Source 2 — MusicBrainz artist genre tags

Source: **[MusicBrainz](https://musicbrainz.org/)** Web Service v2 (keyless).
Billboard carries **no genre**, so we derive genre at the **artist** level from
MusicBrainz artist `tags`. Endpoint:
`GET /ws/2/artist?query=artist:"<name>"&fmt=json&limit=1`, taking the top match.

**Auth / etiquette:** MusicBrainz requires a descriptive `User-Agent` header
(`unwelcomedata/1.0 ( https://github.com/unwelcomedata )`) or it returns 403, and
asks for <= ~1 request/second. We use a **1.1s** delay with retry/backoff.

**Raw tags only.** We store the full tag list per artist (`name` + `count`) as a
JSON string — we do **NOT** bucket into aggregate genres here. The tags are messy
(Taylor Swift → pop, country, synth-pop, dance-pop, 'female vocals', 'icon',
'2000s'); bucketing is `02-clean`'s job.

**Resumable cache.** Every lookup is appended to
`data/raw/mb_artist_cache.jsonl` (one object per line). A re-run loads the cache
and skips artists already fetched, so an interrupted pull resumes.

### Ingest-scope decision — enrich only artists with >= 20 appearances

The lead chart is **genre rank over time**, which only needs genre for artists that
actually matter to the ranking. Pulling genre for all ~11k unique artists at ~1/sec
is multiple hours, so we enrich the artists that cover the bulk of chart-weeks.

We queried the appearance distribution: a `>= 10` threshold still leaves ~6,855
artists (over 5,000), so per the ingest-scope rule we **raised it to
>= 20**, which is ~4,427 artists covering ~85% of all chart
appearances. Each enriched row carries `total_appearances` so `02-clean` knows the
coverage. **Lower-appearance artists are left ungenred for now** — `02-clean`/`03`
will treat them as 'Other/unknown'. This is a deliberate, documented ingest-scope
decision, not a silent drop.

> ⏳ The MusicBrainz pull is the slow part (~1 req/sec, and MB throttles further),
> so the full enrichment set takes a couple of hours. The actual network pull is
> run ahead of time by **`scripts/_pull_mb_cache.py`** (resumable, appends to
> `data/raw/mb_artist_cache.jsonl`). This cell calls the ingest with **`fetch=False`**
> so it simply **consumes the warm cache** and builds the table without blocking —
> any target artist not yet in the cache is kept with `matched=False` (honest
> coverage). Re-run `scripts/_pull_mb_cache.py` to keep filling the cache, then
> re-run this cell to pick up the newly-cached artists.

In [ ]:
# Coverage of the chosen threshold, shown inline.
counts = artist_appearance_counts(hot100)
THRESHOLD = 20
n_unique = len(counts)
sub = counts[counts['total_appearances'] >= THRESHOLD]
cov = sub['total_appearances'].sum() / counts['total_appearances'].sum()
print(f'unique artists (Billboard spelling): {n_unique:,}')
print(f'artists with >= {THRESHOLD} appearances: {len(sub):,} '
      f'({cov:.1%} of all chart appearances)')

In [ ]:
# fetch=False: consume the cache warmed by scripts/_pull_mb_cache.py (no multi-
# hour block). Uncached target artists come back matched=False until the cache fills.
mb = ingest_musicbrainz_genres(
    cfg, hot100, min_appearances=THRESHOLD,
    rate_limit_seconds=1.1, progress_every=100, fetch=False,
)
print(f'\nenrichment rows: {len(mb):,}')
print(f'matched: {int(mb["matched"].sum()):,}  '
      f'unmatched: {int((~mb["matched"]).sum()):,}')
mb.head(10)

### Load the artist→tags result into DuckDB as `mb_artist_genres_raw`
One row per enriched artist. `tags` is a JSON string (round-trips cleanly in
DuckDB 1.5.5); unmatched artists are kept with `matched=false` and empty tags so
coverage stays honest.

In [ ]:
load_to_duckdb(mb, 'mb_artist_genres_raw', con)
n_mb = con.execute('SELECT COUNT(*) FROM mb_artist_genres_raw').fetchone()[0]
print(f'mb_artist_genres_raw: {n_mb:,} rows')

register_source(
    con, 'mb_artist_genres_raw',
    'MusicBrainz Web Service v2 (artist tags)',
    url=cfg['sources']['musicbrainz_artist']['url'],
    license=cfg['sources']['musicbrainz_artist']['license'],
    notes=(
        'Genre derived at the ARTIST level from MusicBrainz crowd-sourced tags '
        '(name + count), stored RAW (not bucketed). Top artist match per '
        'Billboard-spelling name; unmatched artists kept with matched=false.'
    ),
    methodology=(
        'GET /ws/2/artist?query=artist:"<name>"&limit=1, top match. Descriptive '
        'User-Agent required; <= 1 req/sec (1.1s used). Enriched only artists with '
        '>= 20 Hot 100 appearances (~85% of chart-weeks); lower-'
        'appearance artists left ungenred (Other/unknown downstream).'
    ),
    series_breaks=(
        'Tags are crowd-sourced and artist-level, NOT per-song or point-in-time -- '
        'they reflect an artist\'s whole catalog as currently tagged, so a genre '
        'assigned to an early-career chart hit is approximate. Not every artist is '
        'enriched (appearance threshold).'
    ),
)
print('registered mb_artist_genres_raw in _sources')

## Provenance + table summary

In [ ]:
from src.clean_quality import get_sources
print('=== _sources ===')
display(get_sources(con)[['duckdb_table', 'source_name', 'retrieved']])

print('\n=== hot100_raw sample ===')
display(con.execute('SELECT * FROM hot100_raw LIMIT 5').df())

print('=== mb_artist_genres_raw sample (top artists) ===')
display(con.execute(
    'SELECT artist, mb_name, matched, total_appearances, '
    'LEFT(tags, 90) AS tags_preview '
    'FROM mb_artist_genres_raw ORDER BY total_appearances DESC LIMIT 8'
).df())

---
**Next:** `02-clean.ipynb` — type the Hot 100 appearances, normalize artist names, and bucket the raw MusicBrainz tags into aggregate genres.

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')